In [ ]:
from pyspark.sql import functions as F

# 1. Configuração de caminhos
path_gold = "dbfs:/Volumes/workspace/default/data/delta/gold/"
path_output = "dbfs:/Volumes/workspace/default/data/delta/output/"

print("🚀 Iniciando Simulação de Delta Sharing & Resumo Executivo...")

# 2. Carregar Tabelas Gold para Análise
df_customer = spark.read.format("delta").load(f"{path_gold}customer_summary")
df_product = spark.read.format("delta").load(f"{path_gold}product_summary")
df_seller = spark.read.format("delta").load(f"{path_gold}seller_summary")

# --- BLOCO DE EXPORTAÇÃO CSV ---
tables = {"customer": df_customer, "product": df_product, "seller": df_seller}

for name, df in tables.items():
    export_path = f"{path_output}gold_{name}_summary_export.csv"
    df.coalesce(1).write.format("csv").mode("overwrite").option("header", "true").save(export_path)
    print(f"✅ Exportação CSV concluída: {name}")

# --- BLOCO DE RESUMO EXECUTIVO (Requisito image_cf7e81.png) ---
print("\n" + "="*50)
print("📊 RESUMO EXECUTIVO DO E-COMMERCE")
print("="*50)

# Total de clientes únicos
total_customers = df_customer.count()
print(f"👤 Total de clientes únicos atendidos: {total_customers}")

# Receita total consolidada (usando a tabela de produtos para precisão)
total_revenue = df_product.select(F.sum("total_revenue")).collect()[0][0]
print(f"💰 Receita total consolidada: R$ {total_revenue:,.2f}")

# Top 3 categorias de produto por receita
print("\n🏆 Top 3 categorias de produto por receita:")
top_categories = df_product.orderBy(F.desc("total_revenue")).limit(3).select("product_category_name", "total_revenue").collect()
for i, row in enumerate(top_categories, 1):
    print(f"   {i}. {row['product_category_name']}: R$ {row['total_revenue']:,.2f}")

# Estado com maior volume de pedidos
top_state = df_customer.groupBy("customer_state").agg(F.sum("total_orders").alias("vol")).orderBy(F.desc("vol")).first()
print(f"\n📍 Estado com maior volume de pedidos: {top_state['customer_state']} ({top_state['vol']} pedidos)")

# Vendedor com melhor avaliação média (mínimo 10 pedidos)
best_seller = df_seller.filter(F.col("total_orders") >= 10).orderBy(F.desc("avg_review_score")).first()
if best_seller:
    print(f"\n⭐ Melhor Vendedor (min. 10 pedidos): {best_seller['seller_id']}")
    print(f"   Nota Média: {best_seller['avg_review_score']:.2f}")

print("="*50 + "\n🏁 Pipeline Finalizado!")